In [ ]:
# full_camera_calibration_auto_detect.py
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from collections import Counter

# -------------------------
# 1.CONFIGURATION
# -------------------------
# the PNG images from the left camera folder
image_path_glob = '/kaggle/input/datasets/danielwe14/stereocamera-chessboard-pictures/data/imgs/leftcamera/*.png'


square_size = 0.030

# output dir for calibration data
out_dir = '/kaggle/working/calib_auto'
os.makedirs(out_dir, exist_ok=True) 
# keep None for auto-detect
FORCE_CB_SIZE = None  
# grids to try out
candidate_sizes = [
    (6, 5), (7, 6), (8, 5), (9, 6), (10, 7), (11, 8), (12, 9), (5,4), (4,3)
]

# -------------------------
# 2. HELPER FUNCTION: ROBUST CORNER DETECTION
# -------------------------
def detect_for_size(img_bgr, chessboard_size):
    """
    This function tries multiple image processing techniques to find the chessboard corners.
    If standard detection fails, it tries adjusting contrast or resizing the image.
    """
    # convert to gray
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # basic cv2 flags
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE

    # Attempt 1:  try standard grayscale
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, flags)
    if ret:
        return True, corners, "std"

    # Attempt 2:  hist equalization
    gray_eq = cv2.equalizeHist(gray)
    ret2, corners2 = cv2.findChessboardCorners(gray_eq, chessboard_size, flags)
    if ret2:
        return True, corners2, "equalize"

    # Attempt 3:  try different scales
    for scale in (0.5, 0.75, 1.25, 1.5, 2.0):
       
        small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
        r, c = cv2.findChessboardCorners(small, chessboard_size, flags)
        if r:
            # fix coordinates
            c_scaled = c / scale
            return True, c_scaled, f"scaled_{scale}"

    return False, None, "none"

# -------------------------
# 3. STEP 1: SCAN IMAGES & AUTO-SELECT BEST GRID SIZE
# -------------------------

images = sorted(glob.glob(image_path_glob))
if len(images) == 0:
    raise RuntimeError(f"No images found at {image_path_glob}. Please check path.")

# grid size counter
size_counts = Counter()

for fname in images:
    img = cv2.imread(fname)
    if img is None:
        continue
        
    
    for s in candidate_sizes:
        ok, corners, mode = detect_for_size(img, s)
        if ok:
           
            size_counts[s] += 1
            break  

#  selecting the grid size with highest number of successful detections
if FORCE_CB_SIZE is not None:
    chosen_size = FORCE_CB_SIZE
else:
    
    chosen_size = max(size_counts, key=lambda k: size_counts[k])
    print(f"Auto-selected optimal inner-corner size: {chosen_size} (Detected in {size_counts[chosen_size]} images)")

# -------------------------
# 4. STEP 2: EXTRACT REFINED CORNERS USING CHOSEN SIZE
# -------------------------
cb_width, cb_height = chosen_size
chessboard_size = (cb_width, cb_height)



objp = np.zeros((cb_width * cb_height, 3), np.float32)
objp[:, :2] = np.mgrid[0:cb_width, 0:cb_height].T.reshape(-1, 2)
objp *= square_size

objpoints = [] # 3D real-world points 
imgpoints = [] # 2D image pixel points
plotted_first = False
failed = []

#  actual data extraction
for fname in images:
    img = cv2.imread(fname)
    if img is None:
        continue
        
  
    ok, corners, mode = detect_for_size(img, chessboard_size)
    if not ok:
        failed.append(fname) 
        continue

    # refining 
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    corners = corners.astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    corners_refined = cv2.cornerSubPix(gray, corners, (5, 5), (-1, -1), criteria)

    
    objpoints.append(objp.copy())
    imgpoints.append(corners_refined)

    
    if not plotted_first:
        vis = img.copy()
        cv2.drawChessboardCorners(vis, chessboard_size, corners_refined, True)
        save_vis = os.path.join(out_dir, f'first_detection_{cb_width}x{cb_height}.png')
        cv2.imwrite(save_vis, vis)
        plotted_first = True

# -------------------------
# 5. STEP 3:  CAMERA CALIBRATION ALGORITHM
# -------------------------
if len(objpoints) < 3:
    raise RuntimeError("Not enough detections to calibrate. Need at least 3 images.")

#  resolution  image (Width, Height)
image_size = (img.shape[1], img.shape[0])

#  FUNCTION: Calculates  intrinsic matrix, distortion coeffs, and extrinsic vectors
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, image_size, None, None)

# Calculate  mean reprojection error of all images 
total_error = 0
for i in range(len(objpoints)):
   
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
   e
    err = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    total_error += err
mean_error = total_error / len(objpoints)

# -------------------------
# 6.  OUTPUT
# -------------------------
print("\n" + "="*60)
print(" 📷 CAMERA CALIBRATION RESULTS 📷 ")
print("="*60)
print(f"Total Images Processed : {len(images)}")
print(f"Successful Detections  : {len(objpoints)}")
print(f"Grid Size Used         : {cb_width} x {cb_height}")

print("\n--- 1. INTRINSIC CAMERA MATRIX (K) ---")

np.set_printoptions(formatter={'float': lambda x: f"{x:10.2f}"})
print(mtx)
np.set_printoptions() 

print(f"\nExtracted Intrinsic Parameters:")
print(f" • Focal Length (fx)     : {mtx[0,0]:.4f} px")
print(f" • Focal Length (fy)     : {mtx[1,1]:.4f} px")
print(f" • Principal Point (cx)  : {mtx[0,2]:.4f} px")
print(f" • Principal Point (cy)  : {mtx[1,2]:.4f} px")
print(f" • Axis Skew (gamma)     : {mtx[0,1]:.4f}")

print("\n--- 2. LENS DISTORTION COEFFICIENTS ---")
print(f" • Radial (k1)           : {dist[0][0]:.6f}")
print(f" • Radial (k2)           : {dist[0][1]:.6f}")
print(f" • Tangential (p1)       : {dist[0][2]:.6f}")
print(f" • Tangential (p2)       : {dist[0][3]:.6f}")
print(f" • Radial (k3)           : {dist[0][4]:.6f}")

print("\n--- 3. ALGORITHM ACCURACY ---")
print(f" • Overall RMS Error     : {ret:.4f} pixels")
print(f" • Mean Reprojection Err : {mean_error:.4f} pixels")
print("="*60 + "\n")


savefile = os.path.join(out_dir, 'camera_calib.npz')
np.savez(savefile, camera_matrix=mtx, dist_coeff=dist, rvecs=rvecs, tvecs=tvecs, image_size=image_size)


und = cv2.undistort(img, mtx, dist, None)
und_path = os.path.join(out_dir, 'undistorted_example.png')
cv2.imwrite(und_path, und)
print(f"Files saved successfully to: {out_dir}")

Auto-selected optimal inner-corner size: (10, 7) (Detected in 8 images)

 📷 CAMERA CALIBRATION RESULTS 📷 
Total Images Processed : 20
Successful Detections  : 8
Grid Size Used         : 10 x 7

--- 1. INTRINSIC CAMERA MATRIX (K) ---
[[    682.42       0.00     526.80]
 [      0.00     692.35     279.75]
 [      0.00       0.00       1.00]]

Extracted Intrinsic Parameters:
 • Focal Length (fx)     : 682.4215 px
 • Focal Length (fy)     : 692.3455 px
 • Principal Point (cx)  : 526.8034 px
 • Principal Point (cy)  : 279.7522 px
 • Axis Skew (gamma)     : 0.0000

--- 2. LENS DISTORTION COEFFICIENTS ---
 • Radial (k1)           : 0.020713
 • Radial (k2)           : -0.104993
 • Tangential (p1)       : 0.001274
 • Tangential (p2)       : 0.000971
 • Radial (k3)           : 0.096085

--- 3. ALGORITHM ACCURACY ---
 • Overall RMS Error     : 0.1676 pixels
 • Mean Reprojection Err : 0.0200 pixels

Files saved successfully to: /kaggle/working/calib_auto
